### RAG pipelines- Data Ingestion to Vector DB piplines 

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\manas_978n0hd\OneDrive\Desktop\Full stack app development\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
pdf_dir = Path("../data")
print(pdf_dir.resolve())        # shows absolute path
print(list(pdf_dir.rglob("*"))) # shows ALL files found

C:\Users\manas_978n0hd\OneDrive\Desktop\Full stack app development\RAG\data
[WindowsPath('../data/pdf_files'), WindowsPath('../data/text_files'), WindowsPath('../data/vector_store'), WindowsPath('../data/pdf_files/embeddings.pdf'), WindowsPath('../data/pdf_files/NIPS-2017-attention-is-all-you-need-Paper.pdf'), WindowsPath('../data/pdf_files/pdf1.pdf'), WindowsPath('../data/pdf_files/pdf2.pdf'), WindowsPath('../data/pdf_files/pdf3.pdf'), WindowsPath('../data/pdf_files/pdf4.pdf'), WindowsPath('../data/text_files/machine_learning.txt'), WindowsPath('../data/text_files/python_intro.txt'), WindowsPath('../data/vector_store/27df36ca-1cfb-47dd-8b42-15824cc506fc'), WindowsPath('../data/vector_store/40d1e6d2-bf82-481e-a55a-f4bcb765468c'), WindowsPath('../data/vector_store/8bf30a7d-f2df-44d8-b398-396bc360b745'), WindowsPath('../data/vector_store/chroma.sqlite3'), WindowsPath('../data/vector_store/ffd8a730-b15a-4fae-a1d7-0ff36a0fc905'), WindowsPath('../data/vector_store/27df36ca-1cfb-47dd-8b42-15

In [3]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in directory"""

    all_documents =[]
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print (pdf_files)

    print(f" Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing : {pdf_file.name} ")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:   
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")

    return all_documents

all_pdf_documents = process_all_pdfs("../data")

[WindowsPath('../data/pdf_files/embeddings.pdf'), WindowsPath('../data/pdf_files/NIPS-2017-attention-is-all-you-need-Paper.pdf'), WindowsPath('../data/pdf_files/pdf1.pdf'), WindowsPath('../data/pdf_files/pdf2.pdf'), WindowsPath('../data/pdf_files/pdf3.pdf'), WindowsPath('../data/pdf_files/pdf4.pdf')]
 Found 6 PDF files to process
Processing : embeddings.pdf 
  ✓ Loaded 27 pages
Processing : NIPS-2017-attention-is-all-you-need-Paper.pdf 
  ✓ Loaded 11 pages
Processing : pdf1.pdf 
  ✓ Loaded 11 pages
Processing : pdf2.pdf 
  ✓ Loaded 11 pages
Processing : pdf3.pdf 
  ✓ Loaded 11 pages
Processing : pdf4.pdf 
  ✓ Loaded 11 pages


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:)', 'creationdate': '2025-09-01T00:50:53+00:00', 'author': 'Peng Yu; En Xu; Bin Chen; Haibiao Chen; Yinfei Xu', 'doi': 'https://doi.org/10.48550/arXiv.2508.21632', 'keywords': '', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'moddate': '2025-09-01T00:50:53+00:00', 'title': 'QZhou-Embedding Technical Report', 'arxivid': 'https://arxiv.org/abs/2508.21632v1', 'source': '..\\data\\pdf_files\\embeddings.pdf', 'total_pages': 27, 'page': 0, 'page_label': '1', 'source_file': 'embeddings.pdf', 'file_type': 'pdf'}, page_content='QZhou-Embedding Technical Report\n Kingsoft AI\nQZhou-Embedding Technical Report\nPeng Yu, En Xu, Bin Chen, Haibiao Chen, Yinfei Xu\nKingsoft AI ∗\nAugust 2025\nAbstract\nWe present QZhou-Embedding, a general-purpose contextual text embed-\nding model with exceptional text representation capabilit ies. Built upon the\nQwen2.5-7B-Instruct foundation model, we designed a uniﬁe 

In [5]:
### Text splitting get into chunks

def split_documents(documents, chunk_size= 500, chunk_overlap=100):
    """Split Documents into smaller chunks for better RAG performance."""
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("\n Example of Chunks")
        print(f"Content1: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs


In [6]:
chunks = split_documents(all_pdf_documents)

Split 82 documents into 775 chunks

 Example of Chunks
Content1: QZhou-Embedding Technical Report
 Kingsoft AI
QZhou-Embedding Technical Report
Peng Yu, En Xu, Bin Chen, Haibiao Chen, Yinfei Xu
Kingsoft AI ∗
August 2025
Abstract
We present QZhou-Embedding, a genera
Metadata: {'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:)', 'creationdate': '2025-09-01T00:50:53+00:00', 'author': 'Peng Yu; En Xu; Bin Chen; Haibiao Chen; Yinfei Xu', 'doi': 'https://doi.org/10.48550/arXiv.2508.21632', 'keywords': '', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'moddate': '2025-09-01T00:50:53+00:00', 'title': 'QZhou-Embedding Technical Report', 'arxivid': 'https://arxiv.org/abs/2508.21632v1', 'source': '..\\data\\pdf_files\\embeddings.pdf', 'total_pages': 27, 'page': 0, 'page_label': '1', 'source_file': 'embeddings.pdf', 'file_type': 'pdf'}


### Embedding and vector store DB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generations using SentenceTransformer"""

    def __init__(self, model_name: str= "all-MiniLM-L6-V2"):
        """Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer model"""
        try:
            print(f"loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape(len(texts),embedding_dimensions)
        """

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddinggs for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embedding with shape: {embeddings.shape}")
        return embeddings

## Initialise the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager 


loading embedding model: all-MiniLM-L6-V2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2776.01it/s]


Model loaded successfully. Embedding dimension: 384


### Vectore Store

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Delete existing collection to avoid duplicates on re-run
            try:
                self.client.delete_collection(name=self.collection_name)
                print("Cleared existing collection")
            except Exception:
                  pass

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    
    

Cleared existing collection
Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [10]:
### Convert the text into embeddings

texts= [doc.page_content for doc in chunks]

## Generate Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## Store in vector Database
vectorstore.add_documents(chunks,embeddings)


Generating embeddinggs for 775 texts...


Batches: 100%|██████████| 25/25 [00:15<00:00,  1.58it/s]


Generated embedding with shape: (775, 384)
Adding 775 documents to vector store...
Successfully added 775 documents to vector store
Total documents in collection: 775


## Retriever Pipeline from Vector Store

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("tell me about machine translation in attention is all you need pdf?")

Retrieving documents for query: 'tell me about machine translation in attention is all you need pdf?'
Top K: 5, Score threshold: 0.0
Generating embeddinggs for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 72.32it/s]

Generated embedding with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_3eac5175_250',
  'content': 'For translation tasks, the Transformer can be trained signiﬁcantly faster than architectures based\non recurrent or convolutional layers. On both WMT 2014 English-to-German and WMT 2014\nEnglish-to-French translation tasks, we achieve a new state of the art. In the former task our best\nmodel outperforms even all previously reported ensembles.\nWe are excited about the future of attention-based models and plan to apply them to other tasks. We',
  'metadata': {'subject': 'Neural Information Processing Systems http://nips.cc/',
   'creator': 'PyPDF',
   'source': '..\\data\\pdf_files\\NIPS-2017-attention-is-all-you-need-Paper.pdf',
   'source_file': 'NIPS-2017-attention-is-all-you-need-Paper.pdf',
   'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)',
   'title': 'Attention is All you Need',
   'creationdate': '',
   'producer': 'PyPDF2',
   'language': 'en-US',
   'published': 

In [14]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddinggs for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 48.69it/s]

Generated embedding with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_44cecb7c_40',
  'content': '(InfoNCE[48], Cosent[49], etc.).\nOur design principle aims to accommodate more tasks and data types , enabling cross-\ndomain and cross-task data to eﬀectively enhance embedding capa bilities. We propose\na uniﬁed multi-task learning framework that categorizes training da ta into three task\ntypes: retrieval, NLI, and classiﬁcation, with customized data and training solutions\nfor each, allowing most natural text data to be converted into emb edding training data',
  'metadata': {'moddate': '2025-09-01T00:50:53+00:00',
   'page_label': '5',
   'total_pages': 27,
   'creator': 'arXiv GenPDF (tex2pdf:)',
   'page': 4,
   'license': 'http://creativecommons.org/licenses/by/4.0/',
   'keywords': '',
   'title': 'QZhou-Embedding Technical Report',
   'creationdate': '2025-09-01T00:50:53+00:00',
   'author': 'Peng Yu; En Xu; Bin Chen; Haibiao Chen; Yinfei Xu',
   'content_length': 463,
   'source_file': 'embeddings.pdf',
   'producer': 'pikepdf 8.15.1

### RAG Pipeline- VectorDB To LLM Output Generation

In [20]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
class GroqLLM:
    def __init__(self, model_name: str = "llama-3.1-8b-instant", api_key: str =None):
        """
        Initialize Groq LLM

        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.getenv("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")

        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context

        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length

        Returns:
            Generated response string
        """

        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

            Context: {context}

            Question: {question}

            Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )

        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)

        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content

        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting

        Args:
            query: User question
            context: Retrieved context

        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

        Question: {query}

        Answer:"""

        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

